# EDA

Initial exploration notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import yaml
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

# Charger les paramètres centralisés
with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

print("✅ Imports OK")
print(f"📁 Dataset cible : {params['data']['raw_path']}")

In [ ]:
df = pd.read_csv(f"../{params['data']['raw_path']}")

df.shape

In [ ]:
import pandas as pd

# 1. Calcul des métriques
types = df.dtypes
nulls = df.isnull().sum()
percents = (df.isnull().sum() / len(df) * 100).round(2)
uniques = df.nunique()
exemples = df.iloc[0]

# 2. Assemblage propre avec concat
audit = pd.concat([types, nulls, percents, uniques, exemples], axis=1)

# 3. Renommage des colonnes
audit.columns = ['Type', 'Nulls', 'Null_%', 'Uniques', 'Exemple']

# 4. Affichage stylisé
print("=" * 60)
print("INVENTAIRE STRUCTUREL")
print("=" * 60)
print(audit)

In [ ]:
# Adapte 'Catégorie' au nom exact de ta colonne cible
TARGET_COL = 'Catégorie'   # ← vérifie ce nom dans ton dataset

print("=" * 60)
print(f"DISTRIBUTION DE LA CIBLE : {TARGET_COL}")
print("=" * 60)

dist = df[TARGET_COL].value_counts()
dist_pct = df[TARGET_COL].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    'Count'   : dist,
    'Pct_%'   : dist_pct.round(2),
    'Ratio vs majoritaire' : (dist.max() / dist).round(2)
})
print(summary.to_string())

# Ratio de déséquilibre global
ratio = dist.max() / dist.min()
print(f"\n⚠️  Ratio déséquilibre max/min : {ratio:.1f}x")
if ratio > 5:
    print("   → Déséquilibre SÉVÈRE — SMOTE nécessaire")
elif ratio > 2:
    print("   → Déséquilibre MODÉRÉ — class_weight='balanced' suffira peut-être")
else:
    print("   → Dataset relativement équilibré")

In [ ]:
import os
import matplotlib.pyplot as plt

# 1. Gestion automatique du dossier de sauvegarde
output_dir = '../notebooks/figures/'
os.makedirs(output_dir, exist_ok=True) # Crée le dossier s'il n'existe pas
output_path = os.path.join(output_dir, '01_distribution_cible.png')

# 2. Configuration du style et des subplots
plt.style.use('seaborn-v0_8-muted') # Style plus moderne
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- PARTIE GAUGHE : Barplot ---
dist.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white', alpha=0.9)
axes[0].set_title('Distribution des Catégories (Effectifs)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Nombre de tickets', fontsize=11)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

# Ajout des étiquettes au-dessus des barres
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width()/2., p.get_height()), 
                     ha='center', va='center', 
                     xytext=(0, 9), 
                     textcoords='offset points',
                     fontsize=10, fontweight='bold')

# --- PARTIE DROITE : Pie Chart ---
# Couleurs harmonisées
colors = plt.cm.tab10(range(len(dist))) 
dist.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=140, 
          colors=colors, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Répartition en %', fontsize=14, fontweight='bold', pad=15)
axes[1].set_ylabel('') # Supprime le label par défaut 'None'

# 3. Finalisation et Sauvegarde
plt.tight_layout()
try:
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"✅ Figure sauvegardée avec succès ici : {output_path}")
except Exception as e:
    print(f"❌ Erreur lors de la sauvegarde : {e}")

plt.show()

In [ ]:
# Adapte ces noms à tes colonnes réelles
TITLE_COL = 'Titre'
DESC_COL  = 'Description'

# Fusionner pour analyse
df['text_combined'] = (
    df[TITLE_COL].fillna('') + ' ' + df[DESC_COL].fillna('')
).str.strip()

# Longueurs
df['len_title']    = df[TITLE_COL].fillna('').apply(len)
df['len_desc']     = df[DESC_COL].fillna('').apply(len)
df['len_combined'] = df['text_combined'].apply(len)
df['word_count']   = df['text_combined'].apply(lambda x: len(x.split()))

print("=" * 60)
print("STATISTIQUES DE LONGUEUR DES TEXTES")
print("=" * 60)
print(df[['len_title', 'len_desc', 'len_combined', 'word_count']].describe().round(2))

# Tickets quasi-vides
empty_thresh = 5  # moins de 5 mots = quasi-vide
quasi_vides  = df[df['word_count'] < empty_thresh]
print(f"\n⚠️  Tickets quasi-vides (< {empty_thresh} mots) : {len(quasi_vides)}")
if len(quasi_vides) > 0:
    print(quasi_vides[[TITLE_COL, DESC_COL, 'word_count']].head(10).to_string())

In [ ]:
# Visualisation des longueurs
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label in zip(axes,
                           ['len_title', 'len_desc', 'word_count'],
                           ['Longueur Titre', 'Longueur Description', 'Nombre de mots']):
    df[col].hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(label, fontsize=12)
    ax.axvline(df[col].median(), color='red', linestyle='--', label=f'Médiane: {df[col].median():.0f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../notebooks/figures/02_longueurs_textes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 60)
print("ANALYSE DES DUPLICATAS")
print("=" * 60)

# Duplicatas exacts sur le texte combiné
n_dupl = df.duplicated(subset=['text_combined']).sum()
print(f"Duplicatas exacts (texte combiné) : {n_dupl}")

# Duplicatas sur le titre uniquement
n_dupl_title = df.duplicated(subset=[TITLE_COL]).sum()
print(f"Titres identiques                : {n_dupl_title}")

# Top titres répétés
print(f"\nTop 10 titres les plus répétés :")
print(df[TITLE_COL].value_counts().head(10).to_string())

In [ ]:
print("=" * 60)
print("COHÉRENCE DES LABELS PAR TITRE RÉPÉTÉ")
print("=" * 60)

# Pour chaque titre qui apparaît plusieurs fois,
# vérifier si la catégorie est toujours la même
title_cats = (
    df.groupby(TITLE_COL)[TARGET_COL]
    .nunique()
    .reset_index()
    .rename(columns={TARGET_COL: 'n_categories_differentes'})
)

incoherents = title_cats[title_cats['n_categories_differentes'] > 1]
print(f"Titres identiques avec catégories différentes : {len(incoherents)}")

if len(incoherents) > 0:
    print("\n⚠️  Exemples d'incohérences :")
    exemples = df[df[TITLE_COL].isin(incoherents[TITLE_COL])]\
               [[TITLE_COL, TARGET_COL]]\
               .sort_values(TITLE_COL)
    print(exemples.head(20).to_string())

In [ ]:
# Adapte ces noms à tes colonnes réelles
STATUS_COL     = 'STATUT'          # ← vérifie
RESOLUTION_COL = 'Résolution/Motif'      # ← vérifie (ou 'Motif', 'Solution'...)

print("=" * 60)
print("AUDIT BASE DE CONNAISSANCE RAG")
print("=" * 60)

# Valeurs uniques du statut
print(f"Valeurs uniques de '{STATUS_COL}' :")
print(df[STATUS_COL].value_counts().to_string())

# Filtrer les tickets résolus avec résolution non-vide
mask_resolu  = df[STATUS_COL].str.lower().str.contains('r.solu|closed|fermé', na=False)
mask_solution = df[RESOLUTION_COL].notna() & (df[RESOLUTION_COL].str.strip() != '')

rag_base = df[mask_resolu & mask_solution]

print(f"\n✅ Tickets résolus avec résolution renseignée : {len(rag_base)}")
print(f"   ({len(rag_base)/len(df)*100:.1f}% du dataset total)")

if len(rag_base) < 200:
    print("⚠️  ALERTE : Base RAG trop petite — les recommandations seront limitées")
elif len(rag_base) < 500:
    print("⚠️  Base RAG acceptable mais perfectible")
else:
    print("✅  Base RAG de bonne taille")

# Distribution par catégorie dans la base RAG
print(f"\nDistribution RAG par catégorie :")
print(rag_base[TARGET_COL].value_counts().to_string())

In [ ]:
# Créer un rapport texte de l'audit
rapport = f"""
========================================
RAPPORT D'AUDIT EDA — HELPDESK AI
========================================
Date           : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
Dataset        : {params['data']['raw_path']}
Nb tickets     : {len(df)}
Nb colonnes    : {df.shape[1]}

--- CIBLE ---
Colonne cible  : {TARGET_COL}
Nb classes     : {df[TARGET_COL].nunique()}
Ratio déséquil : {ratio:.1f}x

--- TEXTE ---
Mots médiane   : {df['word_count'].median():.0f}
Tickets vides  : {len(quasi_vides)}
Duplicatas     : {n_dupl}
Labels incoh.  : {len(incoherents)}

--- RAG ---
Base RAG size  : {len(rag_base)} tickets
"""

with open('../notebooks/rapport_audit.txt', 'w', encoding='utf-8') as f:
    f.write(rapport)

print(rapport)
print("✅ Rapport sauvegardé dans notebooks/rapport_audit.txt")

In [ ]:
git add notebooks/01_eda.ipynb notebooks/rapport_audit.txt notebooks/figures/
